# S23DR 2026 — Procedural Roof Reconstruction

Geometry-first pipeline: no neural network weights needed.

**Steps**
1. Install dependencies & pull repo
2. Load validation dataset
3. Run procedural pipeline on one sample (debug)
4. Full evaluation: HSS over all 1024 validation samples
5. Visualise: point cloud + GT + procedural wireframe

In [ ]:
!pip install -q datasets huggingface_hub scipy numpy shapely

import os
REPO_DIR = "/content/3d_building_construction"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import io, zipfile
import numpy as np
from datasets import load_dataset
from s23dr.metrics import hss
from s23dr.procedural import reconstruct_to_segments

print("Imports OK")

In [ ]:
HF_DATASET = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT      = "validation"

def _unpack(row):
    out = {}
    with zipfile.ZipFile(io.BytesIO(row["data"])) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out

rows = [_unpack(r) for r in load_dataset(HF_DATASET, split=SPLIT)]
print(f"Loaded {len(rows)} validation samples")
print(f"Keys in sample 0: {list(rows[0].keys())}")

In [ ]:
# ── Diagnostic: understand point cloud fields ─────────────────────────────────
r0 = rows[0]
xyz = r0["xyz_norm"]
vf  = r0["vote_frac"]
cid = r0["class_id"]

print("=== vote_frac distribution ===")
for thr in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    n = (vf >= thr).sum()
    print(f"  >= {thr:.1f} : {n:4d}  ({100*n/len(vf):.1f}%)")

print("\n=== class_id distribution ===")
unique, counts = np.unique(cid, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  class {u:3d} : {c:4d} pts  ({100*c/len(cid):.1f}%)")

print("\n=== z distribution ===")
z = xyz[:, 2]
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d} : {np.percentile(z, p):.4f}")

print("\n=== XY distance from origin ===")
xy_dist = np.sqrt(xyz[:,0]**2 + xyz[:,1]**2)
for p in [0, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d} : {np.percentile(xy_dist, p):.4f}")

print(f"\n=== Available keys ===")
for k, v in r0.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}, min={v.min():.3f}, max={v.max():.3f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── Debug: run pipeline on one sample ──────────────────────────────────────
import time

SAMPLE_IDX = 0
r = rows[SAMPLE_IDX]

xyz      = r["xyz_norm"]              # (N, 3)
vf       = r["vote_frac"]             # (N,) view-agreement score
cid      = r["class_id"]             # (N,)
src      = r["source"]               # (N,) target-building mask
gt_segs  = r["gt_segments"]          # (E, 2, 3)

print(f"Total points      : {len(xyz)}")
print(f"vote_frac > 0     : {(vf > 0).sum()}  ({100*(vf>0).mean():.1f}%)")
print(f"vote_frac >= 0.3  : {(vf >= 0.3).sum()}  ({100*(vf>=0.3).mean():.1f}%)")

t0 = time.time()
pred_segs = reconstruct_to_segments(xyz, vote_frac=vf, class_id=cid, source=src)
elapsed = time.time() - t0

scores = hss(pred_segs, gt_segs)

print(f"\norder_id     : {r['order_id']}")
print(f"GT segments  : {len(gt_segs)}")
print(f"Pred segments: {len(pred_segs)}")
print(f"Time         : {elapsed*1000:.0f} ms")
print(f"HSS          : {scores['hss']:.4f}")
print(f"Precision    : {scores['precision']:.4f}")
print(f"Recall       : {scores['recall']:.4f}")

In [ ]:
# ── Full evaluation over all 1024 validation samples ───────────────────────
import time

results = []
t0 = time.time()

for i, r in enumerate(rows):
    xyz     = r["xyz_norm"]
    vf      = r["vote_frac"]
    cid     = r["class_id"]
    src     = r["source"]
    gt_segs = r["gt_segments"]

    pred_segs = reconstruct_to_segments(xyz, vote_frac=vf, class_id=cid, source=src)
    scores    = hss(pred_segs, gt_segs)
    results.append({"order_id": r["order_id"], **scores})

    if (i + 1) % 100 == 0:
        mean_h = np.mean([r["hss"] for r in results])
        print(f"  [{i+1:4d}/1024]  mean_hss={mean_h:.4f}  ({time.time()-t0:.0f}s)")

hss_arr  = np.array([r["hss"]       for r in results])
prec_arr = np.array([r["precision"] for r in results])
rec_arr  = np.array([r["recall"]    for r in results])

print("=" * 45)
print(f"  Samples   : {len(results)}")
print(f"  HSS       : {hss_arr.mean():.4f}  (std {hss_arr.std():.4f})")
print(f"  Precision : {prec_arr.mean():.4f}")
print(f"  Recall    : {rec_arr.mean():.4f}")
print("=" * 45)

In [ ]:
# ── Visualise: point cloud + GT wireframe + procedural wireframe ─────────────
import plotly.graph_objects as go

SAMPLE_IDX  = 0      # ← change to inspect different samples
VOTE_THRESH = 0.3    # same threshold used by the pipeline
MAX_PTS     = 2048

r         = rows[SAMPLE_IDX]
xyz_np    = r["xyz_norm"]
vf        = r["vote_frac"]
src       = r["source"]           # target-building mask (1 = target house)
gt_segs   = r["gt_segments"]
pred_segs = reconstruct_to_segments(xyz_np, vote_frac=vf, class_id=r["class_id"], source=src)
scores    = hss(pred_segs, gt_segs)

# Apply the same two filters the pipeline uses internally:
#   source == 1  → keep only the target building, drop neighbours
#   vote_frac >= VOTE_THRESH  → multi-view-agreed points only
src_mask  = src == 1
src_voted = src_mask & (vf >= VOTE_THRESH)
xyz_vis   = xyz_np[src_voted]

rng = np.random.default_rng(0)
vis = rng.choice(len(xyz_vis), min(MAX_PTS, len(xyz_vis)), replace=False)
pc  = xyz_vis[vis]

pc_trace = go.Scatter3d(
    x=pc[:,0], y=pc[:,1], z=pc[:,2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.45),
    name=f"Source+voted pts ({src_voted.sum()})",
)

def _seg_trace(segs, color, name, width=4):
    if len(segs) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode="lines",
                            line=dict(color=color, width=width), name=name)
    xs, ys, zs = [], [], []
    for s in segs:
        xs += [float(s[0,0]), float(s[1,0]), None]
        ys += [float(s[0,1]), float(s[1,1]), None]
        zs += [float(s[0,2]), float(s[1,2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color=color, width=width), name=name)

fig = go.Figure(data=[
    pc_trace,
    _seg_trace(gt_segs,   "limegreen", f"GT ({len(gt_segs)} segs)"),
    _seg_trace(pred_segs, "red",       f"Procedural ({len(pred_segs)} segs)"),
])
fig.update_layout(
    title=(f"Sample {SAMPLE_IDX} | {r['order_id']} | "
           f"HSS={scores['hss']:.3f}  P={scores['precision']:.3f}  R={scores['recall']:.3f}"),
    scene=dict(aspectmode="data"),
    height=680, margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()

print(f"Cloud : {src_voted.sum()} pts  (source==1 & vf>={VOTE_THRESH})")
print(f"Excl  : {(~src_mask).sum()} background pts, {(src_mask & (vf < VOTE_THRESH)).sum()} low-vote pts")
print(f"GT    : {len(gt_segs)} segs  |  Pred: {len(pred_segs)} segs")

In [ ]:
# ── Multi-τ breakdown: HSS at tighter match thresholds ──────────────────────
# τ is the max allowed Hausdorff distance (in normalised coords) for a segment
# to count as "matched".  The competition default is τ=0.20, but checking
# tighter values reveals whether the shapes actually align or just overlap loosely.

taus = [0.05, 0.10, 0.15, 0.20, 0.30]

r         = rows[SAMPLE_IDX]
xyz_np    = r["xyz_norm"]
vf        = r["vote_frac"]
src       = r["source"]
gt_segs   = r["gt_segments"]
pred_segs = reconstruct_to_segments(xyz_np, vote_frac=vf, class_id=r["class_id"], source=src)

print(f"Sample {SAMPLE_IDX}  |  GT={len(gt_segs)} segs  |  Pred={len(pred_segs)} segs")
print(f"{'tau':>6}  {'HSS':>6}  {'Precision':>9}  {'Recall':>7}")
print("-" * 35)
for tau in taus:
    s = hss(pred_segs, gt_segs, tau=tau)
    marker = "  ← competition default" if tau == 0.20 else ""
    print(f"{tau:>6.2f}  {s['hss']:>6.3f}  {s['precision']:>9.3f}  {s['recall']:>7.3f}{marker}")

print()
print("τ = max Hausdorff distance for a segment to count as matched (normalised coords)")
print("A large gap between τ=0.05 and τ=0.20 means shapes overlap loosely but don't align precisely.")

In [ ]:
# ── Visualise: surface normal segmentation ────────────────────────────────────
import plotly.graph_objects as go
import numpy as np
from s23dr.procedural.preprocess import estimate_normals, _ransac_ground_plane

SAMPLE_IDX        = 0
WALL_PITCH_THRESH = 55.0
GROUND_PITCH_THRESH = 20.0
GROUND_EPS        = 0.04
XY_RADIUS         = 0.38

r   = rows[SAMPLE_IDX]
src = r['source']
vf  = r['vote_frac']

xyz_all = r['xyz_norm']
src_mask = src == 1
xyz = xyz_all[src_mask] if src_mask.sum() >= 10 else xyz_all
voted = vf[src_mask] >= 0.3 if src_mask.sum() >= 10 else vf >= 0.3
if voted.sum() >= 10:
    xyz = xyz[voted]

print(f'Points after source+vote filter: {len(xyz)}')

normals   = estimate_normals(xyz, k=12)
pitch_deg = np.degrees(np.arccos(np.clip(np.abs(normals[:, 2]), 0.0, 1.0)))
is_wall = pitch_deg > WALL_PITCH_THRESH
horizontal = pitch_deg < GROUND_PITCH_THRESH
h_pts      = xyz[horizontal]
is_ground  = np.zeros(len(xyz), dtype=bool)
if len(h_pts) >= 15:
    result = _ransac_ground_plane(h_pts, eps=GROUND_EPS)
    if result is not None:
        gn, gd = result
        dists  = np.abs(xyz @ gn - gd)
        is_ground = (dists < GROUND_EPS) & horizontal
else:
    z_thresh  = np.percentile(xyz[:, 2], 30)
    is_ground = (xyz[:, 2] < z_thresh) & horizontal

xy_dist  = np.sqrt(xyz[:, 0]**2 + xyz[:, 1]**2)
is_far   = xy_dist > XY_RADIUS
is_roof  = ~is_wall & ~is_ground & ~is_far

print(f'  Wall:   {is_wall.sum()}')
print(f'  Ground: {is_ground.sum()}')
print(f'  Far XY: {is_far.sum()}')
print(f'  Roof:   {is_roof.sum()}')

def _scatter(pts, color, name, size=2, opacity=0.7):
    if len(pts) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode='markers', name=name,
                            marker=dict(size=size, color=color, opacity=opacity))
    return go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
                        marker=dict(size=size, color=color, opacity=opacity), name=name)

fig = go.Figure(data=[
    _scatter(xyz[is_wall],   'red',       f'Wall ({is_wall.sum()})'),
    _scatter(xyz[is_ground], 'sienna',    f'Ground ({is_ground.sum()})'),
    _scatter(xyz[is_far & ~is_wall & ~is_ground], 'lightgrey',
             f'Far XY ({(is_far & ~is_wall & ~is_ground).sum()})'),
    _scatter(xyz[is_roof],   'limegreen', f'Roof ({is_roof.sum()})'),
])

gt_segs = r['gt_segments']
xs, ys, zs = [], [], []
for s in gt_segs:
    xs += [float(s[0,0]), float(s[1,0]), None]
    ys += [float(s[0,1]), float(s[1,1]), None]
    zs += [float(s[0,2]), float(s[1,2]), None]
fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
    line=dict(color='dodgerblue', width=3), name=f'GT ({len(gt_segs)} segs)'))

fig.update_layout(
    title=f"Sample {SAMPLE_IDX} | {r['order_id']} — surface normal segmentation",
    scene=dict(aspectmode='data'),
    height=700, margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()
